# Exercise 1: Understand the data of the print farm
**Internet of Production – AI, week 1** · about 55 min · work in pairs

**Scenario:** Three 3D printers produce parts of a teaching kit. Each printer logs its state **every minute**; disturbances are logged separately.

**Goal:** Load the data of one printer, clean it, and find out what is special about it. At the end you look for a **warning sign before a disturbance**: this is the starting point of exercise 2. Every `___` is a gap you fill. The check questions are answered in the text cells.

| File | Content |
|---|---|
| `Printer_0/1/2.csv` | State, temperatures, filament sensor, energy counter (per minute) |
| `Printer_0/1/2_error.csv` | Time and type of each disturbance |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('../../00_Data/printfarm')   # folder with the CSV files
PRINTER = 1                               # start with printer 1

## Step 1: A first look (naively)
Run the cell. What is wrong?

In [ ]:
raw = pd.read_csv(DATA / f'Printer_{PRINTER}.csv')
print(raw.shape)
raw.head()

## Step 2: Load the file correctly
Open the file in a text editor (first lines). Which **separator** and which **decimal sign** are used? Check `pd.read_csv(..., sep=..., decimal=...)`.

In [ ]:
df = pd.read_csv(DATA / f'Printer_{PRINTER}.csv', sep=';', decimal=',')
print(df.shape)
print(df.dtypes)
df.head()

## Step 3: Are the files complete and regular?
Check **all three printers**: number of rows, number of rows with data (`dropna(how='all')`), period, and whether the time stamps are exactly one minute apart (gaps).

In [ ]:
for k in (0, 1, 2):
    d = pd.read_csv(DATA / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False)
    t = pd.to_datetime(d['Timestamp'].dropna(), format='%d.%m.%Y %H:%M')
    gaps = int((t.diff().dropna() != pd.Timedelta(minutes=1)).sum())
    print(f'printer {k}: rows {len(d):7d}, with data {len(d.dropna(how="all")):7d}, {t.min()} -> {t.max()}, gaps: {gaps}')

**Your observation:**

All three files contain 713,742 rows with data, one per minute from 5 Jan 2024 08:18 to 14 May 2025 23:59, without gaps and without empty rows. The regular sampling allows rolling windows over minutes without further preparation. (Check such properties anyway: exported files often contain empty rows or gaps.)

## Step 4: Clean and parse the time stamp
Drop empty rows, convert `Timestamp` (format `day.month.year hour:minute`) into a datetime and check the sampling interval.

In [ ]:
df = df.dropna(how='all').copy()
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d.%m.%Y %H:%M')
df = df.set_index('Timestamp')
print(df.index.min(), '->', df.index.max())
print(df.index.to_series().diff().value_counts().head())

## Step 5: The energy counter
`Total_Power` looks like a measured value. Plot it over time. What kind of quantity is it? Derive the **energy per minute** and compare the states.

In [ ]:
df['Total_Power'].plot(figsize=(10, 3), title='Total_Power')
plt.show()

In [ ]:
df['Energy'] = df['Total_Power'].diff()
print(df.groupby('State')['Energy'].mean().round(5))
print('negative steps:', (df['Energy'] < 0).sum())

**What is `Total_Power`, and which state consumes the most?**

A cumulative counter that only increases (assumed unit: kWh). Consumption per minute is its difference. Printing needs about twice as much energy per minute (about 0.0016 per minute, roughly 96 W if the unit is kWh) as ready or idle (about 0.0008).

## Step 6: States
How are the minutes distributed over the states?

In [ ]:
share = df['State'].value_counts(normalize=True).mul(100).round(1)
print(share)
share.plot.barh(title='Share of minutes per state (%)')
plt.show()

## Step 7: Disturbances
Load the error file of the printer and count the disturbances per type. Then look at the minute data **around the first extruder blockage**: in which state is the printer, and what happens to the extruder temperature?

In [ ]:
err = pd.read_csv(DATA / f'Printer_{PRINTER}_error.csv', sep=';').dropna(how='all')
err['Timestamp'] = pd.to_datetime(err['Timestamp'], format='%d.%m.%Y %H:%M')
print(err['Error'].value_counts())
first = err.loc[err['Error'] == 'Extruder blocked', 'Timestamp'].iloc[0]
print('first extruder blockage:', first)
df.loc[first - pd.Timedelta(minutes=3): first + pd.Timedelta(minutes=3), ['State', 'Temp_Extruder', 'Temp_Bed']]

How long does a printer **stay** in the state `error`? Count the connected episodes and look at their length in minutes.

In [ ]:
episode = (df['State'] != df['State'].shift()).cumsum()
lengths = df.groupby(episode).agg(state=('State', 'first'), minutes=('State', 'size'))
e = lengths[lengths['state'] == 'error']['minutes']
print('episodes in state error:', len(e))
print(e.describe().round(1))

## Step 8: Is there a warning sign before an extruder blockage?
A single minute of the energy consumption scatters strongly (the counter has a resolution of 0.1 Wh). Therefore **average**: take the mean energy per minute **while printing** for each of the 12 hours before an extruder blockage, over all blockages of this printer, and compare it with normal printing.

In [ ]:
printing = df[df['State'] == 'printing']
normal = printing['Energy'].mean() * 1000        # Wh per minute (if Total_Power is in kWh)
blocked = err.loc[err['Error'] == 'Extruder blocked', 'Timestamp']

rows = []
for t in blocked:
    for h in range(12, 0, -1):                   # hour bins: 12 h ... 1 h before the blockage
        w = printing.loc[t - pd.Timedelta(hours=h): t - pd.Timedelta(hours=h - 1, minutes=1), 'Energy']
        if len(w) >= 30:
            rows.append((h, w.mean() * 1000))
profile = pd.DataFrame(rows, columns=['hours_before', 'Wh_per_min']).groupby('hours_before')['Wh_per_min'].mean()

ax = profile.plot(marker='o', figsize=(9, 3.5), color='#003A79', label='before an extruder blockage')
ax.axhline(normal, color='grey', ls='--', label='normal printing')
ax.invert_xaxis(); ax.set_xlabel('hours before the blockage'); ax.set_ylabel('energy per minute (Wh)'); ax.legend(); plt.show()

Do the other disturbance types show the same? Compare the mean energy in the 6 hours before each disturbance type with normal printing.

In [ ]:
for typ, ts in err.groupby('Error')['Timestamp']:
    vals = []
    for t in ts:
        w = printing.loc[t - pd.Timedelta(hours=6): t - pd.Timedelta(minutes=1), 'Energy']
        if len(w) >= 30:
            vals.append(w.mean() * 1000)
    print(f'{typ:24s} n = {len(vals):2d}   energy in the 6 h before: {np.mean(vals):.3f} Wh/min   (normal: {normal:.3f})')

**What do you see? Why do we need the average, and which disturbance types show it?**

Before an extruder blockage the mean energy per minute while printing rises step by step over the last hours, from the normal level of about 1.645 Wh/min to about 1.72 Wh/min directly before the blockage (about +4 %): the extruder needs slightly more power. A single minute scatters by about 0.14 Wh, so the trend only shows up in the average. Connection interrupted and empty filament show no change. The rise could serve as an early warning (exercise 2).

## Step 9: Make it reusable
Wrap your cleaning steps into a function. We use it again in week 3.

In [ ]:
def load_printer(k, data=DATA):
    """Minute data of printer k: clean, indexed by time, with the column Energy (per minute)."""
    d = pd.read_csv(data / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='%d.%m.%Y %H:%M')
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    return d

def load_errors(k, data=DATA):
    e = pd.read_csv(data / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    return e

p2 = load_printer(2)
print(p2.shape, p2.index.min(), p2.index.max())

## Your findings (three to four sentences)
What did you notice about the data? What would have gone wrong if you had used it unchecked?

1) The files use semicolon and decimal comma, and printers 0 and 2 are padded with about 335,000 empty rows (Excel row limit). 2) `Total_Power` is a cumulative counter; only its difference is the consumption, and printing needs about twice the energy of idle states. 3) A disturbance starts with the state `error`, in which the printer stays for many minutes: the state is therefore a consequence of the disturbance and must not be used as a feature to predict it. 4) Before an extruder blockage of printer 1 the energy per minute rises by about 4 %: a candidate for an early warning.